### Improving Model performance through *`Hyperparameter Tuning`*

**Baseline Result:**
Using raw data with slight feature engineering resulted in\
`Root Mean Squared Error = 4.32 seconds`\
`Root Absolute Error     = 3.39 seconds`

**Advance Feature Engineering:**
Using improved *feature engineering* like tire cliff and Cumulative Weather resulted in\
`Root Mean Squared Error = 2.79 seconds`\
`Root Absolute Error     = 2.37 seconds`

In [142]:
import os
import pandas as pd
from dotenv import load_dotenv

import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [143]:
load_dotenv(override=True)

file_path = os.getenv('LOAD_TRAIN_PATH')

if not file_path:
    file_path = '/content/train.csv'

In [144]:
df = pd.read_csv(file_path)

In [145]:
df.head()

,TyreLife,AirTemp,TrackTemp,Rainfall,time_delta,TrackTemp_Rolling4,Rainfall_Rolling4,TyreLife_Sq,Compound_0,Compound_1,Compound_2,Compound_3,Compound_4
0,2.0,16.7,26.2,True,11.414,26.200,1.0,4.0,True,False,False,False,False
1,3.0,16.4,24.8,True,9.624,25.500,2.0,9.0,True,False,False,False,False
2,4.0,16.2,24.6,False,8.666,25.200,2.0,16.0,True,False,False,False,False
3,5.0,16.3,24.3,False,8.279,24.975,2.0,25.0,True,False,False,False,False
4,6.0,16.5,24.1,False,9.395,24.450,1.0,36.0,True,False,False,False,False


In [146]:
X = df.drop(columns=['time_delta', 'TyreLife'])
y = df['time_delta']

In [147]:
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f'Training on {len(X_train)} historical laps.')
print(f'Testing on {len(X_test)} historical laps.')

Training on 10936 historical laps.
Testing on 2734 historical laps.


In [148]:
tscv = TimeSeriesSplit(n_splits=3)

In [149]:
param_distribution = {
    'n_estimators': [300, 450, 600, 800],
    'learning_rate': [0.01, 0.02, 0.03, 0.04],
    'max_depth': [2, 3, 4],
    'subsample': [0.55, 0.65, 0.75, 0.85],
    'colsample_bytree': [0.55, 0.65, 0.75, 0.85],
    'min_child_weight': [4, 5, 7, 10],
    'reg_lambda': [0.5, 1.0, 2.0, 4.0],
    'reg_alpha': [0.0, 0.01, 0.05, 0.1]
}

In [150]:
xgb = XGBRegressor(random_state=42)

In [151]:
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_distribution,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=1
)

In [152]:
search.fit(X_train, y_train)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma...
                   n_iter=30, n_jobs=1,
                   param_distributions={'colsample_bytree': [0.55, 0.65, 0.75,
                                                             0.85],
                                        'learning_rate': [0.01, 0.02, 0.03,
                                                          0.04],
                                        'max_depth': [2, 3, 4],
                                        'min_child_weight': [4, 5, 7, 10],
                                        'n_estimators': [300, 450, 600, 800],
                                        'reg_alpha': [0.0, 0.01, 0.05, 0.1],
                                        'reg_lambda': [0.5, 1.0, 2.0, 4.0],
                                        'subsample': [0.55, 0.65, 0.75, 0.85]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [153]:
print(f'Best Parameters: {search.best_params_}')

Best Parameters: {'subsample': 0.65, 'reg_lambda': 1.0, 'reg_alpha': 0.01, 'n_estimators': 600, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.55}


In [154]:
best_model = search.best_estimator_

In [155]:
y_pred = best_model.predict(X_test)

In [156]:
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)

print(f'Tuned Root Mean Squared Error (rmse) = {rmse:.6f}')
print(f'Tuned Mean Absolute Error (mae)      = {mae:.6f}')

Tuned Root Mean Squared Error (rmse) = 2.627063
Tuned Mean Absolute Error (mae)      = 2.259319
